In [49]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import (
    OneHotEncoder,
    LabelEncoder
)

from sklearn.compose import ColumnTransformer

from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)

print("Libraries imported.")

Libraries imported.


In [50]:
PRODUCTS_PATH = (
    r"G:\PMS_Optimization\Preprocess\Preprocessed ORANGEBOOK\products.csv"
)

products = pd.read_csv(
    PRODUCTS_PATH,
    dtype=str
)

print(
    "Dataset:",
    products.shape
)

Dataset: (48502, 18)


In [51]:
features = [
    "Ingredient",
    "DF;Route",
    "Strength",
    "Appl_Type",
    "RLD",
    "RS",
    "Type",
    "Dosage_Form",
    "Route_Of_Administration",
    "Approved_Prior_To_1982"
]

target = "TE_Code"

model_df = products[
    features + [target]
].copy()

print(
    "Model dataset:",
    model_df.shape
)

Model dataset: (48502, 11)


In [52]:
model_df[target] = (
    model_df[target]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.upper()
)

model_df = model_df[
    model_df[target] != ""
].copy()

print(
    "After target cleaning:",
    model_df.shape
)

print(
    model_df[target]
    .value_counts()
)

After target cleaning: (21853, 11)
TE_Code
AB                 14894
AP                  3781
AA                  1043
AB1                  588
AT                   509
AB2                  300
AB3                  129
AN                   129
AB1,AB2,AB3,AB4       72
AO                    71
AP1                   67
BX                    60
AT1                   43
AP2                   37
AB1,AB2,AB3           36
AB4                   30
AT2                   21
AB1,AB2               12
AB1,AB3               11
AT3                    9
BP                     5
AP3                    2
BD                     2
BS                     1
BC                     1
Name: count, dtype: int64


In [53]:
MIN_CLASS_COUNT = 5

class_counts = (
    model_df[target]
    .value_counts()
)

valid_classes = class_counts[
    class_counts >= MIN_CLASS_COUNT
].index

model_df = model_df[
    model_df[target]
    .isin(valid_classes)
].copy()

print(
    "After rare-class filtering:",
    model_df.shape
)

print(
    "Classes:",
    model_df[target].nunique()
)

After rare-class filtering: (21847, 11)
Classes: 21


In [54]:
boolean_features = [
    "RLD",
    "RS",
    "Approved_Prior_To_1982"
]

for col in boolean_features:

    model_df[col] = (
        model_df[col]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "true": 1,
            "false": 0
        })
        .fillna(0)
    )

print(
    model_df[boolean_features]
    .head()
)

   RLD  RS  Approved_Prior_To_1982
0    0   1                       0
1    1   0                       0
4    0   0                       0
6    0   0                       0
7    0   1                       0


In [55]:
categorical_features = [
    "Ingredient",
    "DF;Route",
    "Strength",
    "Appl_Type",
    "Type",
    "Dosage_Form",
    "Route_Of_Administration"
]

for col in categorical_features:

    model_df[col] = (
        model_df[col]
        .fillna("UNKNOWN")
        .astype(str)
        .str.strip()
        .str.upper()
    )

In [56]:
X = model_df[
    features
].copy()

label_encoder = LabelEncoder()

y = label_encoder.fit_transform(
    model_df[target]
)

print(
    "Features:",
    X.shape
)

print(
    "Classes:",
    len(label_encoder.classes_)
)

Features: (21847, 10)
Classes: 21


In [57]:
X_train, X_test, y_train, y_test = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

print(
    "Training:",
    len(X_train)
)

print(
    "Testing:",
    len(X_test)
)

Training: 17477
Testing: 4370


In [58]:
categorical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

numerical_transformer = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        )
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            categorical_transformer,
            categorical_features
        ),

        (
            "numerical",
            numerical_transformer,
            boolean_features
        )
    ]
)

print(
    "Preprocessor ready."
)

Preprocessor ready.


Logistic Regression

In [59]:
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced"
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

lr_train_pred = logistic_model.predict(
    X_train
)

lr_test_pred = logistic_model.predict(
    X_test
)

print(
    "Logistic Regression completed."
)

Logistic Regression completed.


In [60]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=20,
                min_samples_leaf=2,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

rf_model.fit(
    X_train,
    y_train
)

rf_train_pred = rf_model.predict(
    X_train
)

rf_test_pred = rf_model.predict(
    X_test
)

print(
    "Random Forest completed."
)

Random Forest completed.


In [61]:
from xgboost import XGBClassifier

xgb_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=8,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                objective="multi:softmax",
                eval_metric="mlogloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_model.fit(
    X_train,
    y_train
)

xgb_train_pred = xgb_model.predict(
    X_train
)

xgb_test_pred = xgb_model.predict(
    X_test
)

print(
    "XGBoost completed."
)

XGBoost completed.


In [62]:
from lightgbm import LGBMClassifier

lgbm_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),

        (
            "classifier",
            LGBMClassifier(
                n_estimators=300,
                learning_rate=0.05,
                max_depth=10,
                num_leaves=31,
                min_child_samples=20,
                subsample=0.8,
                colsample_bytree=0.8,
                random_state=42,
                n_jobs=-1,
                verbosity=-1
            )
        )
    ]
)

lgbm_model.fit(
    X_train,
    y_train
)

lgbm_train_pred = lgbm_model.predict(
    X_train
)

lgbm_test_pred = lgbm_model.predict(
    X_test
)

print(
    "LightGBM completed."
)

LightGBM completed.


In [63]:
%pip install catboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [64]:
from catboost import CatBoostClassifier

X_cat = X.copy()

for col in categorical_features:

    X_cat[col] = (
        X_cat[col]
        .fillna("UNKNOWN")
        .astype(str)
    )

X_cat_train, X_cat_test, y_cat_train, y_cat_test = (
    train_test_split(
        X_cat,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

cat_indices = [
    X_cat.columns.get_loc(col)
    for col in categorical_features
]

catboost_model = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function="MultiClass",
    eval_metric="TotalF1",
    random_seed=42,
    l2_leaf_reg=5,
    verbose=100
)

catboost_model.fit(
    X_cat_train,
    y_cat_train,
    cat_features=cat_indices,
    eval_set=(
        X_cat_test,
        y_cat_test
    ),
    early_stopping_rounds=50
)

cat_train_pred = (
    catboost_model
    .predict(X_cat_train)
    .astype(int)
    .ravel()
)

cat_test_pred = (
    catboost_model
    .predict(X_cat_test)
    .astype(int)
    .ravel()
)

print(
    "CatBoost completed."
)

0:	learn: 0.8404554	test: 0.8415572	best: 0.8415572 (0)	total: 1.77s	remaining: 14m 44s
100:	learn: 0.9506383	test: 0.9556319	best: 0.9556319 (100)	total: 5m 6s	remaining: 20m 10s
200:	learn: 0.9547886	test: 0.9594215	best: 0.9594215 (199)	total: 10m 39s	remaining: 15m 50s


KeyboardInterrupt: 

In [66]:
results = []

def evaluate_model(
    name,
    y_train_true,
    y_train_pred,
    y_test_true,
    y_test_pred
):

    train_accuracy = accuracy_score(
        y_train_true,
        y_train_pred
    )

    test_accuracy = accuracy_score(
        y_test_true,
        y_test_pred
    )

    precision = precision_score(
        y_test_true,
        y_test_pred,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test_true,
        y_test_pred,
        average="weighted",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_test_true,
        y_test_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_test_true,
        y_test_pred,
        average="weighted",
        zero_division=0
    )

    gap = (
        train_accuracy -
        test_accuracy
    )

    results.append({
        "Model": name,
        "Train_Accuracy": train_accuracy,
        "Test_Accuracy": test_accuracy,
        "Precision": precision,
        "Recall": recall,
        "Macro_F1": macro_f1,
        "Weighted_F1": weighted_f1,
        "Train_Test_Gap": gap
    })


evaluate_model(
    "Logistic Regression",
    y_train,
    lr_train_pred,
    y_test,
    lr_test_pred
)

evaluate_model(
    "Random Forest",
    y_train,
    rf_train_pred,
    y_test,
    rf_test_pred
)

evaluate_model(
    "XGBoost",
    y_train,
    xgb_train_pred,
    y_test,
    xgb_test_pred
)

evaluate_model(
    "LightGBM",
    y_train,
    lgbm_train_pred,
    y_test,
    lgbm_test_pred
)

"""evaluate_model(
    "CatBoost",
    y_cat_train,
    cat_train_pred,
    y_cat_test,
    cat_test_pred
)"""

results_df = pd.DataFrame(
    results
)

display(
    results_df.sort_values(
        "Macro_F1",
        ascending=False
    )
)

,Model,Train_Accuracy,Test_Accuracy,Precision,Recall,Macro_F1,Weighted_F1,Train_Test_Gap
2,XGBoost,0.976426,0.963387,0.957204,0.963387,0.487929,0.959837,0.013039
0,Logistic Regression,0.919265,0.902288,0.944859,0.902288,0.484027,0.919938,0.016977
1,Random Forest,0.784002,0.780092,0.920370,0.780092,0.387879,0.835122,0.003910
3,LightGBM,0.594553,0.597025,0.454753,0.597025,0.036063,0.516266,-0.002472


In [68]:
results_sorted = results_df.sort_values(
    [
        "Macro_F1",
        "Weighted_F1"
    ],
    ascending=False
)

best_model_name = (
    results_sorted
    .iloc[1]["Model"]
)

print(
    "Best model based on Macro F1:",
    best_model_name
)

display(
    results_sorted
)

Best model based on Macro F1: Logistic Regression


,Model,Train_Accuracy,Test_Accuracy,Precision,Recall,Macro_F1,Weighted_F1,Train_Test_Gap
2,XGBoost,0.976426,0.963387,0.957204,0.963387,0.487929,0.959837,0.013039
0,Logistic Regression,0.919265,0.902288,0.944859,0.902288,0.484027,0.919938,0.016977
1,Random Forest,0.784002,0.780092,0.920370,0.780092,0.387879,0.835122,0.003910
3,LightGBM,0.594553,0.597025,0.454753,0.597025,0.036063,0.516266,-0.002472


In [69]:
print("=" * 70)
print("OVERFITTING / GENERALIZATION CHECK")
print("=" * 70)

for _, row in results_df.iterrows():

    gap = row["Train_Test_Gap"]

    print(
        f"\n{row['Model']}"
    )

    print(
        f"Train Accuracy : "
        f"{row['Train_Accuracy']:.4f}"
    )

    print(
        f"Test Accuracy  : "
        f"{row['Test_Accuracy']:.4f}"
    )

    print(
        f"Train-Test Gap : "
        f"{gap:.4f}"
    )

    if gap > 0.10:

        print(
            "⚠️ Possible overfitting"
        )

    elif (
        row["Train_Accuracy"] < 0.75
        and
        row["Test_Accuracy"] < 0.75
    ):

        print(
            "⚠️ Possible underfitting"
        )

    else:

        print(
            "✅ Reasonable generalization"
        )

OVERFITTING / GENERALIZATION CHECK

Logistic Regression
Train Accuracy : 0.9193
Test Accuracy  : 0.9023
Train-Test Gap : 0.0170
✅ Reasonable generalization

Random Forest
Train Accuracy : 0.7840
Test Accuracy  : 0.7801
Train-Test Gap : 0.0039
✅ Reasonable generalization

XGBoost
Train Accuracy : 0.9764
Test Accuracy  : 0.9634
Train-Test Gap : 0.0130
✅ Reasonable generalization

LightGBM
Train Accuracy : 0.5946
Test Accuracy  : 0.5970
Train-Test Gap : -0.0025
⚠️ Possible underfitting


In [70]:
if best_model_name == "Logistic Regression":
    
    final_true = y_test
    final_pred = lr_test_pred

elif best_model_name == "Random Forest":

    final_true = y_test
    final_pred = rf_test_pred

elif best_model_name == "XGBoost":

    final_true = y_test
    final_pred = xgb_test_pred

elif best_model_name == "LightGBM":

    final_true = y_test
    final_pred = lgbm_test_pred

elif best_model_name == "CatBoost":

    final_true = y_cat_test
    final_pred = cat_test_pred


print(
    classification_report(
        final_true,
        final_pred,
        target_names=label_encoder.classes_,
        zero_division=0
    )
)

                 precision    recall  f1-score   support

             AA       0.93      0.99      0.96       209
             AB       1.00      0.90      0.95      2979
            AB1       0.57      0.60      0.58       118
        AB1,AB2       0.00      0.00      0.00         2
    AB1,AB2,AB3       0.00      0.00      0.00         7
AB1,AB2,AB3,AB4       0.42      0.53      0.47        15
        AB1,AB3       0.00      0.00      0.00         2
            AB2       0.25      0.58      0.35        60
            AB3       0.36      0.54      0.43        26
            AB4       0.00      0.00      0.00         6
             AN       1.00      1.00      1.00        26
             AO       1.00      1.00      1.00        14
             AP       0.97      1.00      0.98       756
            AP1       0.64      0.69      0.67        13
            AP2       0.29      0.29      0.29         7
             AT       0.73      1.00      0.85       102
            AT1       0.83    

In [73]:
from sklearn.metrics import classification_report

print(f"\nClassification Report - {best_model_name}\n")

print(
    classification_report(
        y_test,
        y_pred_best,
        labels=best_model.classes_,
        zero_division=0
    )
)


Classification Report - Logistic Regression



NameError: name 'y_pred_best' is not defined

In [72]:
import os
import joblib

MODEL_DIR = "../models"

os.makedirs(
    MODEL_DIR,
    exist_ok=True
)

if best_model_name == "Logistic Regression":

    best_model = logistic_model

elif best_model_name == "Random Forest":

    best_model = rf_model

elif best_model_name == "XGBoost":

    best_model = xgb_model

elif best_model_name == "LightGBM":

    best_model = lgbm_model

elif best_model_name == "CatBoost":

    best_model = catboost_model


model_path = os.path.join(
    MODEL_DIR,
    "best_TE_model.pkl"
)

joblib.dump(
    best_model,
    model_path
)

print(
    "Best model saved:"
)

print(model_path)

Best model saved:
../models\best_TE_model.pkl
